# Fitting Methods
This notebook describes various methods available in the gwrefpy package for fitting observation and reference data.

Currently supported fitting methods include:
- Linear Regression
- Nth order polynomial fitting
- Chebyshev polynomial fitting

# Linear Regression
The linear regression fitting method fits a straight line to the data using the least squares method. It is suitable for data that exhibits a linear relationship.

The equation for a linear regression is given by:

$$y = a_0 + a_1x$$

where $a_0$ and $a_1$ are the coefficients.

# Nth Order Polynomial Fitting
The Nth order polynomial fitting method fits a polynomial of degree N to the data. This method is useful when more degrees of freedom are needed to capture the relationship between the variables.

The equation for a Nth order polynomial is given by:

$$y = a_0 + a_1 x + a_2 x^2 + ... + a_N x^N$$

where $a_0$, $a_1$, $...$, $a_N$ are the coefficients of the polynomial.

# Chebyshev Polynomial Fitting
The Chebyshev polynomial fitting method uses Chebyshev polynomials to fit the data. Chebyshev polynomials are orthogonal polynomials that can provide a good approximation for functions over a specific interval.

The equation for a Chebyshev polynomial of degree N is given by:

$$y = a_0T_0(x) + a_1T_1(x) + a_2T_2(x) + ... + a_NT_n(x)$$

where $T_N(x)$ is the Chebyshev polynomial of degree $N$ and $a_0$, $a_1$, $...$, $a_N$ are the coefficients of the polynomial. The Chebyshev polynomials are defined recursively as:

$$T_0(x) = 1$$

$$T_1(x) = x$$

$$T_{n+1}(x) = 2xT_n(x) - T_{n-1}(x)$$

## Prediction Confidence Widths

Each fitting method calculates a prediction width, the half-width of its prediction interval. Given a fitted value $\hat{y}$ and width $w$, the interval is

$$
[\hat{y} - w,\; \hat{y} + w].
$$

Here, $p$ is the requested confidence level and $n$ is the number of paired observations after time-equivalent grouping and the selected calibration window (`tmin` and `tmax`). The t critical value is written as $t_{(1+p)/2,\,\nu}$, where $\nu$ is the degrees of freedom used by the method.

### Linear Regression

For linear regression, residuals are $r_i = y_i - \hat{y}_i$, and the implementation calculates the residual standard error as

$$
s_{\mathrm{lin}} = \sqrt{\frac{1}{n-2}\left(\sum_{i=1}^{n}r_i^2 - \frac{\left[\sum_{i=1}^{n}r_i(x_i-\bar{x})\right]^2}{\sum_{i=1}^{n}(x_i-\bar{x})^2}\right)}.
$$

It then uses a single width for every reference value:

$$
w_{\mathrm{lin}} = t_{(1+p)/2,\,n-1}\,s_{\mathrm{lin}}\sqrt{1+\frac{1}{n}}.
$$

Thus, this implementation's linear prediction width is constant across the reference series. The residual standard error uses $n-2$ in its calculation, while the t critical value uses $n-1$ degrees of freedom.

### Nth-Order Polynomial Fit

For a degree-$d$ polynomial, the basis vector at reference value $x$ is

$$
\boldsymbol{\phi}(x) = [x^d,\,x^{d-1},\,\ldots,\,x,\,1]^\mathsf{T}.
$$

Let $X$ be the design matrix whose rows are the basis vectors for the grouped training reference values. The residual standard error and degrees of freedom are

$$
s = \sqrt{\frac{\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}{n-d-1}},
\qquad \nu = n-d-1.
$$

For a new reference value $x_*$, its leverage and prediction width are

$$
h(x_*) = \boldsymbol{\phi}(x_*)^\mathsf{T}(X^\mathsf{T}X)^{-1}\boldsymbol{\phi}(x_*),
\qquad
w(x_*) = t_{(1+p)/2,\,\nu}\,s\sqrt{1+h(x_*)}.
$$

The leverage term makes the interval wider for values that are farther from the training reference values in the polynomial design. The implementation scales the design-matrix columns when calculating the inverse-equivalent operator for numerical stability, and rejects rank-deficient designs.

### Chebyshev Polynomial Fit

The Chebyshev method uses the same residual standard error, degrees of freedom, leverage, and width equations as the Nth-order polynomial method. Its basis vector is instead

$$
\boldsymbol{\phi}(x) = [T_0(x),\,T_1(x),\,\ldots,\,T_d(x)]^\mathsf{T},
$$

where $T_k(x)$ is the Chebyshev polynomial of degree $k$. This basis is used to build $X$ and evaluate leverage at each prediction value.

### Evaluating New Reference Values

The polynomial and Chebyshev fits save the leverage operator calculated from their original grouped training data. `FitResultData.get_prediction_constant(ref_series)` uses that saved operator to calculate widths for supplied reference values, including appended values, without adding them to the training window or refitting. The upper and lower bounds are the fitted values plus or minus those widths, respectively.